In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 22 — ENSEMBLES DE MODELOS: O PODER DO COMITÊ DE ESPECIALISTAS

> "Nunca confie no julgamento de um único especialista, por mais brilhante que seja. A verdadeira sabedoria emerge do dissenso e do consenso de um comitê diverso."
> — Um princípio da tomada de decisão robusta

Minha jornada me deixou com vários candidatos fortes: o SVM, geômetra preciso; a Regressão Logística, estatística interpretável; o Gradient Boosting, aprendiz sequencial. Cada um via o problema de um ângulo. A pergunta agora era: eu precisava escolher apenas um?

Pensei em como os hospitais lidam com casos difíceis. Não confiam num único médico: reúnem um comitê — radiologista, oncologista, patologista. Cada um traz sua experiência, eles debatem e chegam a uma decisão conjunta, quase sempre mais robusta que a de qualquer um isolado. E se eu fizesse o mesmo com meus modelos?

## Ensembles de Votação: Combinando Modelos Fortes

Enquanto o Random Forest é um ensemble de modelos **homogêneos** (muitas árvores), um **Voting Ensemble** combina modelos **heterogêneos** — algoritmos diferentes treinados nos mesmos dados. A intuição: se os modelos são diversos, seus erros também são; combinando as previsões, erros individuais tendem a se cancelar.

Há duas formas de combinar:

**Hard Voting** — cada modelo vota numa classe; vence a maioria.

**Soft Voting** — média das *probabilidades* previstas; vence a classe de maior probabilidade média. Geralmente preferível, porque leva em conta a **confiança** de cada modelo: quem prevê "Maligno" com 99% pesa mais que quem prevê "Benigno" com 51%.

Vamos montar um comitê com Regressão Logística, SVM e Gradient Boosting, por Soft Voting, e compará-lo aos membros individuais — medindo acurácia **e** recall maligno.

#### Código 22.1: Construindo e Avaliando o Comitê


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

log = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", random_state=42))])
svm = Pipeline([("sc", StandardScaler()), ("svm", SVC(kernel="rbf", C=10, gamma=0.01,
                                                      probability=True, random_state=42))])
gb  = GradientBoostingClassifier(n_estimators=100, random_state=42)

comite = VotingClassifier([("lr", log), ("svm", svm), ("gb", gb)], voting="soft")

metricas = {"acc": "accuracy", "rec": recall_maligno}
for nome, est in [("Reg. Logistica", log), ("SVM", svm),
                  ("Gradient Boost", gb), ("ENSEMBLE", comite)]:
    r = cross_validate(est, X_train, y_train, cv=5, scoring=metricas)
    print(f"{nome:15s} acuracia {r['test_acc'].mean():.4f} | recall Maligno {r['test_rec'].mean():.4f}")

Reg. Logistica  acuracia 0.9688 | recall Maligno 0.9548
SVM             acuracia 0.9771 | recall Maligno 0.9548
Gradient Boost  acuracia 0.9563 | recall Maligno 0.9265
ENSEMBLE        acuracia 0.9812 | recall Maligno 0.9662


O comitê venceu — de verdade. O **Ensemble** atingiu **98,12%** de acurácia, acima do melhor membro individual (SVM, 97,71%), e — o que mais importa — elevou o **recall maligno para 96,62%**, o melhor de todo o livro. Onde o SVM tem dúvida, a Regressão Logística e o Gradient Boosting muitas vezes têm certeza, e a confiança combinada supera a hesitação individual. Como os erros dos três são, em parte, não correlacionados, combiná-los produziu um modelo mais forte que a soma das partes. Este é o nosso **modelo final**.

> **📌 CHECKLIST DO CAPÍTULO 22**
>
> ✅ Entende um Ensemble de Votação e como difere do Random Forest.
>
> ✅ Sabe a diferença entre **Hard** e **Soft Voting** e por que este costuma ser melhor.
>
> ✅ Executou o Código 22.1 e viu o ensemble superar **todos** os membros (98,12% acc, 96,62% recall maligno).
>
> **⚠️ CRÍTICO:** A chave de um bom ensemble é a **diversidade**. Combinar modelos que abordam o problema de formas fundamentalmente diferentes (linear, geométrico, árvores) tem muito mais chance de sucesso do que combinar variações do mesmo algoritmo.

O sucesso do comitê confirmou a filosofia que permeou todo o meu trabalho: a robustez nasce da diversidade — nos dados, nas *features* e, agora, nos próprios modelos. Eu não tinha mais um único "melhor" modelo, mas um sistema que culminava numa decisão colegiada e mais confiável.

Eu me aproximava do fim da jornada técnica. Mas antes de declará-la completa, precisava voltar aos erros. A matriz de confusão me dizia *quantos* eram; não me dizia *quais*. Eu precisava olhar os pacientes que meu comitê ainda errava, procurar padrões nos seus dados. Precisava fazer uma **Análise de Erros**.
